# Clase 080 — Árboles de decisión: entrenamiento, visualización, CART

Entrenamos un `DecisionTreeClassifier` con el algoritmo **CART** (*greedy*, binario) sobre
**Iris**, calculamos la impureza **Gini** a mano, comparamos **Gini vs. Entropy**, y aprendemos
a **leer** el árbol con `plot_tree`. Cerramos mostrando la limitación de la frontera
**axis-aligned** sobre `make_moons`.

Requiere: `numpy`, `matplotlib`, `scikit-learn`.

## 1. Fit + score baseline

Entrenamos un árbol de profundidad 2 sobre Iris (las 4 features) y consultamos las
probabilidades de la primera flor con `predict_proba`. Los árboles **no requieren escalado**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, make_moons
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score

RND = 42
iris = load_iris()
X, y = iris.data, iris.target

clf = DecisionTreeClassifier(max_depth=2, random_state=RND).fit(X, y)
acc = accuracy_score(y, clf.predict(X))
print(f"accuracy train (max_depth=2): {acc:.3f}")
assert acc >= 0.90, "un arbol depth=2 ya separa muy bien Iris"
print("predict_proba de la primera flor:", np.round(clf.predict_proba(X[:1])[0], 3))

## 2. Impureza Gini a mano

`G = 1 - Σ p_k²`. El nodo raíz de Iris tiene 50/50/50, así que las tres proporciones son 1/3.
Comparamos nuestro cálculo con `tree_.impurity[0]`.

In [ ]:
p = np.array([50, 50, 50]) / 150
gini_manual = 1 - np.sum(p ** 2)
gini_sklearn = clf.tree_.impurity[0]
print(f"Gini calculado a mano : {gini_manual:.4f}")
print(f"Gini de sklearn (raiz): {gini_sklearn:.4f}")
assert np.isclose(gini_manual, gini_sklearn), "deben coincidir"
print("coinciden: la raiz 50/50/50 tiene Gini = 2/3")

## 3. Gini vs. Entropy

En la práctica producen árboles casi idénticos. Entrenamos uno con cada criterio y comparamos
*accuracy* y las features usadas en los splits.

In [ ]:
for crit in ["gini", "entropy"]:
    t = DecisionTreeClassifier(criterion=crit, max_depth=3, random_state=RND).fit(X, y)
    feats = sorted(set(f for f in t.tree_.feature if f >= 0))
    nombres = [iris.feature_names[i] for i in feats]
    print(f"criterion={crit:<8} acc={accuracy_score(y, t.predict(X)):.3f} "
          f"features usadas={nombres}")
print("Gini y Entropy dan arboles equivalentes; Gini es marginalmente mas rapido")

## 4. Visualización con plot_tree

`plot_tree` solo necesita matplotlib (a diferencia de `export_graphviz`, que requiere el binario
`dot`). Leemos el split del root, los thresholds y la clase de cada hoja. `export_text` da la
misma información en texto.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
plot_tree(clf, feature_names=iris.feature_names, class_names=iris.target_names,
          filled=True, rounded=True, ax=ax)
ax.set_title("DecisionTreeClassifier(max_depth=2) sobre Iris")
plt.tight_layout(); plt.show()

print(export_text(clf, feature_names=list(iris.feature_names)))

## 5. Frontera axis-aligned

Cada split de CART es **ortogonal a un eje** (`x_k ≤ t_k`), así que la frontera resultante es
una unión de rectángulos: no puede aprender una diagonal de forma compacta. Lo mostramos con
`make_moons`.

In [ ]:
Xm, ym = make_moons(n_samples=300, noise=0.2, random_state=RND)
tree_m = DecisionTreeClassifier(max_depth=4, random_state=RND).fit(Xm, ym)

x0 = np.linspace(Xm[:, 0].min() - 0.5, Xm[:, 0].max() + 0.5, 300)
x1 = np.linspace(Xm[:, 1].min() - 0.5, Xm[:, 1].max() + 0.5, 300)
xx, yy = np.meshgrid(x0, x1)
Z = tree_m.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
ax.scatter(Xm[:, 0], Xm[:, 1], c=ym, cmap="coolwarm", edgecolor="k", s=18)
ax.set_title("Frontera axis-aligned (max_depth=4): escalones, no diagonal")
plt.tight_layout(); plt.show()

## Ejercicios

1. Entrená `DecisionTreeClassifier(max_depth=2)` sobre Iris y reportá `accuracy` en train y las
   probabilidades de la primera flor con `predict_proba`.
2. Calculá el Gini del nodo raíz (50/50/50) a mano y verificalo contra `tree_.impurity[0]`.
3. Entrená dos árboles `max_depth=3`, uno con `criterion="gini"` y otro con `"entropy"`.
   Compará *accuracy* y las features usadas en los splits.
4. Renderizá el árbol del ejercicio 1 con `plot_tree` e identificá el split del root, sus
   thresholds y la clase predicha por cada hoja.
5. Entrená un árbol `max_depth=4` sobre `make_moons` y graficá la frontera: observá los
   rectángulos axis-aligned.

## Conclusiones

- **CART** es *greedy* y binario: en cada nodo elige el `(feature, threshold)` que minimiza la
  impureza ponderada de los hijos.
- **Gini** (default) y **Entropy** producen árboles casi idénticos; Gini es más rápido.
- Los árboles **no requieren escalado**: son invariantes a transformaciones monótonas de cada
  feature.
- `plot_tree` audita el modelo sin dependencias externas; `export_text` da la versión en texto.
- La frontera es **axis-aligned** (rectángulos): por eso un solo árbol batalla con fronteras
  oblicuas, y los ensambles mejoran tanto.